#NCCU Smart Waste Classification System

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 準備 YOLO 訓練資料 (Prepare YOLO Training Data)

建立 YOLO 所需的資料結構與 `data.yaml` 設定檔。

Create the necessary data structure for YOLO and the `data.yaml` configuration file.

In [ ]:
trash_items = [
    'plastic_bag (塑膠袋)', 'plastic_bottle (塑膠瓶)', 'pet_bottle (寶特瓶)', 'plastic_lunch_box (塑膠餐盒)',
    'cardboard_box (紙箱)', 'paper (紙類)', 'glass_bottle (玻璃瓶)', 'aluminum_can (鋁罐)',
    'milk_carton (牛奶盒)', 'lunch_box (便當盒)', 'paper_cup (紙杯)', 'aluminum_foil_pack (鋁箔包)'
]

category_mapping = {
    'plastic_bag (塑膠袋)': 'Plastic',
    'plastic_bottle (塑膠瓶)': 'Plastic',
    'pet_bottle (寶特瓶)': 'Plastic',
    'plastic_lunch_box (塑膠餐盒)': 'Plastic',
    'cardboard_box (紙箱)': 'Paper',
    'paper (紙類)': 'Paper',
    'glass_bottle (玻璃瓶)': 'Cans & Glass',
    'aluminum_can (鋁罐)': 'Cans & Glass',
    'milk_carton (牛奶盒)': 'Tetra Pak & Milk Cartons',
    'lunch_box (便當盒)': 'Tetra Pak & Milk Cartons',
    'paper_cup (紙杯)': 'Tetra Pak & Milk Cartons',
    'aluminum_foil_pack (鋁箔包)': 'Tetra Pak & Milk Cartons'
}

print("Corrected 12 Trash Items:")
print(trash_items)
print("\nCorrected Category Mapping:")
for item, category in category_mapping.items():
    print(f"{item}: {category}")


Corrected 12 Trash Items:
['plastic_bag (塑膠袋)', 'plastic_bottle (塑膠瓶)', 'pet_bottle (寶特瓶)', 'plastic_lunch_box (塑膠餐盒)', 'cardboard_box (紙箱)', 'paper (紙類)', 'glass_bottle (玻璃瓶)', 'aluminum_can (鋁罐)', 'milk_carton (牛奶盒)', 'lunch_box (便當盒)', 'paper_cup (紙杯)', 'aluminum_foil_pack (鋁箔包)']

Corrected Category Mapping:
plastic_bag (塑膠袋): Plastic
plastic_bottle (塑膠瓶): Plastic
pet_bottle (寶特瓶): Plastic
plastic_lunch_box (塑膠餐盒): Plastic
cardboard_box (紙箱): Paper
paper (紙類): Paper
glass_bottle (玻璃瓶): Cans & Glass
aluminum_can (鋁罐): Cans & Glass
milk_carton (牛奶盒): Tetra Pak & Milk Cartons
lunch_box (便當盒): Tetra Pak & Milk Cartons
paper_cup (紙杯): Tetra Pak & Milk Cartons
aluminum_foil_pack (鋁箔包): Tetra Pak & Milk Cartons


In [ ]:
import yaml

yaml_file_path = '/content/data.yaml'

# 建立 YOLO 所需的設定檔內容
data_yaml = {
    'path': '/content/drive/MyDrive/trash_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(trash_items),
    'names': trash_items
}

# 將設定檔寫入 data.yaml
with open(yaml_file_path, 'w', encoding='utf-8') as file:
    yaml.dump(data_yaml, file, default_flow_style=False, sort_keys=False, allow_unicode=True)

# 印出內容來確認
print(f"Generated {yaml_file_path} contents:")
with open(yaml_file_path, 'r', encoding='utf-8') as file:
    print(file.read())

Generated /content/data.yaml contents:
path: /content/drive/MyDrive/trash_dataset
train: images/train
val: images/val
nc: 12
names:
- plastic_bag (塑膠袋)
- plastic_bottle (塑膠瓶)
- pet_bottle (寶特瓶)
- plastic_lunch_box (塑膠餐盒)
- cardboard_box (紙箱)
- paper (紙類)
- glass_bottle (玻璃瓶)
- aluminum_can (鋁罐)
- milk_carton (牛奶盒)
- lunch_box (便當盒)
- paper_cup (紙杯)
- aluminum_foil_pack (鋁箔包)



## 資料前處理 (Data Preprocessing)

把這些圖片轉換並整理成 YOLOv8 所需要的格式。
(Convert and organize these images into the format required by YOLOv8.)

In [ ]:
# 步驟 1：定義這 12 個類別的原始資料夾路徑
# 請將等號右邊的路徑替換為您 Google Drive 中實際的資料夾位置
raw_data_paths = {
    'plastic_bag (塑膠袋)': '/content/drive/MyDrive/colab/AI導論/Final Project/Plastics/塑膠袋',
    'plastic_bottle (塑膠瓶)': '/content/drive/MyDrive/colab/AI導論/Final Project/Plastics/塑膠蓋',
    'pet_bottle (寶特瓶)': '/content/drive/MyDrive/colab/AI導論/Final Project/Plastics/寶特瓶',
    'plastic_lunch_box (塑膠餐盒)': '/content/drive/MyDrive/colab/AI導論/Final Project/Plastics/塑膠餐盒',
    'cardboard_box (紙箱)': '/content/drive/MyDrive/colab/AI導論/Final Project/Paper/紙箱',
    'paper (紙類)': '/content/drive/MyDrive/colab/AI導論/Final Project/Paper/紙類',
    'glass_bottle (玻璃瓶)': '/content/drive/MyDrive/colab/AI導論/Final Project/Cans & Glass/玻璃',
    'aluminum_can (鋁罐)': '/content/drive/MyDrive/colab/AI導論/Final Project/Cans & Glass/鋁罐',
    'milk_carton (牛奶盒)': '/content/drive/MyDrive/colab/AI導論/Final Project/Tetra pak & Milk Carton/牛奶盒',
    'lunch_box (便當盒)': '/content/drive/MyDrive/colab/AI導論/Final Project/Tetra pak & Milk Carton/便當盒',
    'paper_cup (紙杯)': '/content/drive/MyDrive/colab/AI導論/Final Project/Tetra pak & Milk Carton/紙杯',
    'aluminum_foil_pack (鋁箔包)': '/content/drive/MyDrive/colab/AI導論/Final Project/Tetra pak & Milk Carton/鋁箔包'
}

print("路徑字典已建立")

路徑字典已建立


In [ ]:
import os
import shutil
import random

# 步驟 2：建立 YOLO 目錄結構並進行資料切分 (80% 訓練集, 20% 驗證集)
yolo_dataset_path = '/content/drive/MyDrive/trash_dataset'
os.makedirs(os.path.join(yolo_dataset_path, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_path, 'images/val'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_path, 'labels/train'), exist_ok=True)
os.makedirs(os.path.join(yolo_dataset_path, 'labels/val'), exist_ok=True)

split_ratio = 0.8

print("開始處理圖片並生成 YOLO 格式資料...")
for class_id, (class_name, folder_path) in enumerate(raw_data_paths.items()):
    if not os.path.exists(folder_path):
        print(f"⚠️ 找不到資料夾: {folder_path} (跳過此類別)")
        continue

    images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(images)

    train_size = int(len(images) * split_ratio)
    train_images = images[:train_size]
    val_images = images[train_size:]

    # 處理訓練集與驗證集
    for split, img_list in [('train', train_images), ('val', val_images)]:
        for img_name in img_list:
            # 複製圖片
            src_img = os.path.join(folder_path, img_name)
            dst_img = os.path.join(yolo_dataset_path, f'images/{split}', f"{class_id}_{img_name}")
            shutil.copy(src_img, dst_img)

            # 生成 YOLO 標註檔 (預設為整張圖片都在範圍內，格式: class x_center y_center width height)
            txt_name = os.path.splitext(img_name)[0] + '.txt'
            dst_txt = os.path.join(yolo_dataset_path, f'labels/{split}', f"{class_id}_{txt_name}")
            with open(dst_txt, 'w') as f:
                f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

print(f"\n資料前處理完成！您的 YOLO 格式資料已儲存至：{yolo_dataset_path}")

開始處理圖片並生成 YOLO 格式資料...


KeyboardInterrupt: 

In [ ]:
import yaml

# 將 data.yaml 的路徑更新為您真實的資料集路徑
yaml_file_path = '/content/data.yaml'

with open(yaml_file_path, 'r', encoding='utf-8') as file:
    data_yaml = yaml.safe_load(file)

data_yaml['path'] = '/content/drive/MyDrive/trash_dataset'

with open(yaml_file_path, 'w', encoding='utf-8') as file:
    yaml.dump(data_yaml, file, default_flow_style=False, sort_keys=False, allow_unicode=True)

print("已將 data.yaml 的路徑更新為真實資料集：/content/drive/MyDrive/trash_dataset")

已將 data.yaml 的路徑更新為真實資料集：/content/drive/MyDrive/trash_dataset


##確認資料總量 (Confirm Data Volume)

In [ ]:
import os

yolo_dataset_path = '/content/drive/MyDrive/trash_dataset'
train_dir = os.path.join(yolo_dataset_path, 'images/train')
val_dir = os.path.join(yolo_dataset_path, 'images/val')

try:
    num_train = len(os.listdir(train_dir))
    num_val = len(os.listdir(val_dir))
    print(f"訓練資料 (Training data): {num_train} 張圖片")
    print(f"驗證資料 (Validation data): {num_val} 張圖片")
    print(f"總資料量 (Total data): {num_train + num_val} 張圖片")
except FileNotFoundError:
    print(f"找不到指定的資料夾，請確認路徑 {yolo_dataset_path} 是否正確或已掛載雲端硬碟。")

訓練資料 (Training data): 1459 張圖片
驗證資料 (Validation data): 681 張圖片
總資料量 (Total data): 2140 張圖片


## 訓練 YOLOv8 模型 (Train YOLOv8 Model)

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os

# 載入預訓練的 YOLOv8 模型 (使用最輕量的 nano 版本作為開始)
model = YOLO('yolov8n.pt')

# 設定專案儲存路徑至 Google Drive，避免斷線遺失
project_path = '/content/drive/MyDrive/trash_detection_project'
os.makedirs(project_path, exist_ok=True)

print("開始訓練 YOLOv8 模型...")

# 開始訓練
# 您可以將 epochs 增加到 50 或 100 來獲得更好的準確度
results = model.train(
    data='/content/data.yaml',
    epochs=20,         # 訓練週期，可根據需要調整
    imgsz=640,         # 影像大小
    project=project_path,  # 訓練結果儲存位置
    name='yolov8_training' # 此次訓練的名稱
)

print("訓練完成！模型與訓練結果已儲存至您的 Google Drive。")

## 評估模型成效 (Evaluate Model Performance)

評估模型的精確度、召回率和平均精確度

Evaluate the model's precision, recall, and mean Average Precision (mAP)

In [ ]:
import yaml
import os

# 確保 trash_items 變數已定義，如果沒有則從頭定義 (防止 runtime 重置)
if 'trash_items' not in locals():
    trash_items = [
        'plastic_bag (塑膠袋)', 'plastic_bottle (塑膠瓶)', 'pet_bottle (寶特瓶)', 'plastic_lunch_box (塑膠餐盒)',
        'cardboard_box (紙箱)', 'paper (紙類)', 'glass_bottle (玻璃瓶)', 'aluminum_can (鋁罐)',
        'milk_carton (牛奶盒)', 'lunch_box (便當盒)', 'paper_cup (紙杯)', 'aluminum_foil_pack (鋁箔包)'
    ]

# 重新生成 data.yaml 檔案
yaml_file_path = '/content/data.yaml'

data_yaml = {
    'path': '/content/drive/MyDrive/trash_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': len(trash_items),
    'names': trash_items
}

with open(yaml_file_path, 'w', encoding='utf-8') as file:
    yaml.dump(data_yaml, file, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"✅ 已成功重新生成 data.yaml 檔案於：{yaml_file_path}")

✅ 已成功重新生成 data.yaml 檔案於：/content/data.yaml


In [ ]:
import os
from ultralytics import YOLO

# 載入訓練好的模型權重
best_model_path = '/content/drive/MyDrive/trash_detection_project/yolov8_training-3/weights/best.pt'

if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print("✅ 成功載入訓練好的模型！")

    # 執行模型驗證
    metrics = model.val(data='/content/data.yaml')

    # 顯示關鍵指標
    print(f"\n模型驗證結果 (Model Validation Results):")
    # Removed the line `print(metrics.results_dict)` as it's no longer needed after identifying correct keys
    print(f"  Precision (精確度): {metrics.results_dict['metrics/precision(B)']:.4f}")
    print(f"  Recall (召回率): {metrics.results_dict['metrics/recall(B)']:.4f}")
    print(f"  mAP50 (平均精確度@IoU=0.5): {metrics.results_dict['metrics/mAP50(B)']:.4f}")
    print(f"  mAP50-95 (平均精確度@IoU=0.5-0.95): {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
else:
    print(f"⚠️ 找不到模型檔案: {best_model_path}，請確認模型路徑是否正確。")


✅ 成功載入訓練好的模型！
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Model summary (fused): 73 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 88.4±60.6 MB/s, size: 39.4 KB)
val: Scanning /content/drive/MyDrive/trash_dataset/labels/val.cache... 681 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 681/681 285.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 20.3it/s 2.1s
                   all        681        681      0.932      0.957      0.963      0.963
     plastic_bag (塑膠袋)         80         80      0.963      0.988      0.994      0.993
  plastic_bottle (塑膠瓶)         87         87      0.934      0.981      0.991      0.991
      pet_bottle (寶特瓶)         41         41      0.908          1      0.995      0.995
plastic_lunch_box (塑膠餐盒)        103        103      0.971          1   

## 測試模型推論 (預測新圖片) Test Model Inference (Predict New Images)

In [ ]:
!pip install gradio ultralytics
import gradio as gr
import cv2
import os
from ultralytics import YOLO

# 1. 載入我們訓練好的最佳模型權重
best_model_path = '/content/drive/MyDrive/trash_detection_project/yolov8_training-3/weights/best.pt'

if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print("✅ 成功載入訓練好的模型！")
else:
    print(f"⚠️ 找不到模型檔案: {best_model_path}")

# 2. 定義預測函數 (固定 threshold 並回傳文字結果)
def predict_image(image):
    if image is None:
        return None, "請上傳圖片 / Please upload an image."

    # 將 RGB 轉為 BGR 供 OpenCV/YOLO 使用
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # 設定閾值 (使用 0.1 讓較低信心的結果也能顯示，避免漏判)
    conf_threshold = 0.1
    iou_threshold = 0.45

    # 進行預測
    results = model.predict(source=img_bgr, conf=conf_threshold, iou=iou_threshold)

    # 畫出預測框並轉回 RGB 以供顯示
    res_plotted = results[0].plot()
    img_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

    # 整理文字輸出結果 (包含物品名稱與對應分類)
    boxes = results[0].boxes
    if len(boxes) == 0:
        result_text = "### 🔍 辨識結果 / Detection Results\n\n**⚠️ 未辨識到任何垃圾 / No trash detected.**\n(可能原因：物品不夠清晰，或模型對此物品信心度過低 / Model confidence is too low.)"
    else:
        result_text = "### 🔍 辨識結果 / Detection Results\n\n"
        for i, box in enumerate(boxes):
            class_id = int(box.cls[0].item())
            conf = box.conf[0].item()
            class_name = model.names[class_id]

            # 將 12 種物品對應到 4 大回收分類 (中英雙語)
            category = "Unknown"
            if class_name in ['plastic_bag (塑膠袋)', 'plastic_bottle (塑膠瓶)', 'pet_bottle (寶特瓶)', 'plastic_lunch_box (塑膠餐盒)']:
                category = "Plastic (塑膠類)"
            elif class_name in ['cardboard_box (紙箱)', 'paper (紙類)']:
                category = "Paper (紙類)"
            elif class_name in ['glass_bottle (玻璃瓶)', 'aluminum_can (鋁罐)']:
                category = "Cans & Glass (金屬與玻璃)"
            elif class_name in ['milk_carton (牛奶盒)', 'lunch_box (便當盒)', 'paper_cup (紙杯)', 'aluminum_foil_pack (鋁箔包)']:
                category = "Tetra Pak & Milk Cartons (紙容器類)"

            result_text += f"**{i+1}. 物品 (Item):** {class_name} \n"
            result_text += f"   - **回收分類 (Category):** {category} \n"
            result_text += f"   - **信心度 (Confidence):** {conf:.2f}\n\n"

    return img_rgb, result_text

# 3. 建立 Gradio 介面
iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy", label="Upload Trash Image (上傳垃圾圖片)"),
    outputs=[
        gr.Image(type="numpy", label="Detection Image (辨識圖片)"),
        gr.Markdown(label="Classification Details (詳細分類結果)")
    ],
    title="🗑️ NCCU Smart Waste Classification System (政大智慧垃圾辨識與分類系統)",
    description="**說明 (Instructions):** 上傳圖片後，AI 將自動畫出辨識框，並於右側顯示**物品名稱**與所屬的**回收分類**。\n Upload an image, and the AI will draw bounding boxes and show the **item name** and its **recycling category** in the panel.",
    theme=gr.themes.Soft()
)

# 4. 啟動應用程式
iface.launch(share=True, debug=False)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ 成功載入訓練好的模型！
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21c5ee6088d0e432c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
